In [1]:
#Librerias utilizadas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import time
import math
import glob
from pathlib import Path

In [2]:
def get_ListOfDF(directory:str, verbose:int = 0):
    # Read all the results tables in the directory
    csv_files = glob.glob(directory)

    # Sort the tables by name
    csv_files = sorted(csv_files)

    # Read all the csv files from the directory, and save we save them in a list
    dataframes = [pd.read_csv(f) for f in csv_files]
    names_frames = [Path(csv).stem.split('_')[0] for csv in csv_files]

    #We print the name of the readed tables if it's requiered
    if verbose == 0: print(names_frames)
    if verbose in [0,1]: print("Readed path: ", directory)

    # We print the tables readed
    if verbose == 0:
        for i in range(0, len(csv_files)):
            print(f"|{i}| The table has been read it from the directory: {csv_files[i]}")
        print("-"*80)
    if verbose in [0,1]:
        print(f"{len(dataframes)} tables has been readed.")

    return dataframes, names_frames

#Directory of the results original
directory_original = "./csvs/results/original/*.csv"
directory_addedLS = "./csvs/results/addedLS/*.csv"

#Read all the results of the dataframes
dataframes, names_frames = get_ListOfDF(directory_addedLS, 1)
dataframes_LS, names_frames = get_ListOfDF(directory_original, 0)

Readed path:  ./csvs/results/addedLS/*.csv
51 tables has been readed.
['ARGLINA', 'BARD', 'BEALE', 'BRKMCC', 'BROWNAL', 'BROWNBS', 'BROWNDEN', 'CHNROSNB', 'CLIFF', 'CUBE', 'DENSCHNA', 'DENSCHNC', 'DENSCHND', 'DENSCHNF', 'DIXON3DQ', 'EIGENALS', 'EIGENBLS', 'ENGVAL2', 'FLETCBV2', 'FLETCHCR', 'GENHUMPS', 'HAIRY', 'HEART6LS', 'HELIX', 'HILBERTA', 'HILBERTB', 'HIMMELBB', 'HUMPS', 'JENSMP', 'KOWOSB', 'LOGHAIRY', 'MANCINO', 'MARATOSB', 'MEXHAT', 'PALMER1C', 'PALMER2C', 'PALMER3C', 'PALMER4C', 'PALMER5C', 'PALMER6C', 'PALMER7C', 'PALMER8C', 'POWELLSQ', 'ROSENBR', 'SINEVAL', 'SISSER', 'TOINTQOR', 'VARDIM', 'WATSON', 'YFITU']
Readed path:  ./csvs/results/original/*.csv
|0| The table has been read it from the directory: ./csvs/results/original/ARGLINA_none.csv
|1| The table has been read it from the directory: ./csvs/results/original/BARD_none.csv
|2| The table has been read it from the directory: ./csvs/results/original/BEALE_none.csv
|3| The table has been read it from the directory: ./csvs/res

In [3]:
problem_dimensions = {
    "ARGLINA": 200,
    "BARD": 3,
    "BEALE": 2,
    "BRKMCC": 2,
    "BROWNAL": 200,
    "BROWNBS": 2,
    "BROWNDEN": 4,
    "CHNROSNB": 50,
    "CLIFF": 2,
    "CUBE": 2,
    "DECONVU": 63,
    "DENSCHNA": 2,
    "DENSCHNB": 2,
    "DENSCHNC": 2,
    "DENSCHND": 2,
    "DENSCHNF": 2,
    "DIXON3DQ": 10000,
    "EIGENALS": 2550,
    "EIGENBLS": 2550,
    "ENGVAL2": 3,
    "EXTROSNB": 1000,
    "FLETCBV2": 5000,
    "FLETCHCR": 1000,
    "GENHUMPS": 5000,
    "HAIRY": 2,
    "HEART6LS": 6,
    "HELIX": 3,
    "HILBERTA": 2,
    "HILBERTB": 10,
    "HIMMELBB": 2,
    "HIMMELBH": 2,
    "HUMPS": 2,
    "JENSMP": 2,
    "KOWOSB": 4,
    "LOGHAIRY": 2,
    "MANCINO": 2,
    "MARATOSB": 2,
    "MEXHAT": 2,
    "PALMER1C": 8,
    "PALMER2C": 8,
    "PALMER3C": 8,
    "PALMER4C": 8,
    "PALMER5C": 5, 
    "PALMER6C": 8,
    "PALMER7C": 8,
    "PALMER8C": 8,
    "POWELLSQ": 2,
    "ROSENBR": 2,
    "SINEVAL": 2,
    "SISSER": 2,
    "TOINTQOR": 50,
    "VARDIM": 200,
    "WATSON": 12,
    "YFITU": 3
}

In [4]:
def construct_matrixResults(list_dataframes: list[pd.DataFrame]):
    # We have 5 methods and 5 variables, then in total we have N \times 5 elements in the dataframe
    result_matrix = np.zeros((5, len(list_dataframes), 6)) #(Mesuared Variables, Problems, Methods)

    #Recolection of the information of each method (recolected in rows)
    for i in range(0, len(list_dataframes)):
        convergence = list_dataframes[i]["Archived Convergence"].to_numpy()
        itterations = list_dataframes[i]["iterations"].to_numpy()
        grads = list_dataframes[i]["Last Gradient"].to_numpy()
        exc = list_dataframes[i]["Execution time"].to_numpy()
        ittPSec = list_dataframes[i]["Iterations per Second"].to_numpy()

        #Setting the information in the result matrix
        result_matrix[:, i, :] = [convergence, itterations, grads, exc, ittPSec]

    return result_matrix


#Names of the methods
col_names = ["Oviedo", "Grads", "Random", "Newton", "BFGS", "GDLS"]
column_names_addedLS = ["G", "A", "Q", "R", "B", "N"]

#Order of the variables:
variable_names = ("Archived Convergence", "iterations", "Last gradient", "Execution time", "Iterations per Second")

#Get the matrix of results
result_matrix = construct_matrixResults(dataframes)

def change_orderOfMethods(result_matrix:np.ndarray, neworder:list[int]):
    """The original order of the methods is
        0. NAMGM/Using the AMGM set of vectors
        1. NAMGM/Using the Queue set of vectors
        2. NAMGM/Using the Random set of vectors
        3. Modified Newton's method
        4. BFGS method
        5. Gradient Descent

    Remark. The use of line search (LS) does not modify the order, this is the given order for all the different kind of experiments
    with or without the use of LS.
    """

    assert len(neworder) == result_matrix.shape[2], "The given reorder does not match with the dimension of the methods"
    reOrdered_matrix = result_matrix[:, :, neworder]
    return reOrdered_matrix

#Reorder the matrix results
result_matrix = change_orderOfMethods(result_matrix, [5, 0, 1, 2, 4, 3])

#Configuration of the printing 
default_values = [None, 1000, -np.inf, np.inf, np.inf]

def isRepeteatedValue(value, array:np.ndarray):
    counter = np.sum(array == value)
    if counter > 1: return True
    else: return False


def printRAWInfo(columnToPrint:int, addProblemName:bool = False, addProblemDim: bool = False, show_info:bool = False,
                 dimBoundLower:int = 0, dimBoundUpper:int = np.inf):
    
    #Assert that the column to be printed is valid
    assert columnToPrint in [1,2,3,4], "Not valid entry to print in the result matrix"
    
    
    #Print the information
    printed_values = 0 
    max_intValue = default_values[columnToPrint]
    for (i, problem) in enumerate(names_frames):
        problemDim = problem_dimensions[problem]
        if dimBoundLower <= problemDim <= dimBoundUpper:
            s = ""
            if addProblemName: s += problem + " & "
            if addProblemDim and addProblemName: s += f"{str(problemDim)} & "
            if columnToPrint in [1, 2, 3]:
                best_value_index = result_matrix[columnToPrint, i, :].argmin()
            else:
                best_value_index = result_matrix[columnToPrint, i, :].argmax()
            for j in range(0, 6):
                value = result_matrix[columnToPrint, i, j]
                best_value = result_matrix[columnToPrint, i, best_value_index]
                repetition_of_values = isRepeteatedValue(best_value, result_matrix[columnToPrint, i, :])
                if value == 0.0: 
                    exponent = decimal = 0
                    if value == best_value and value != max_intValue and repetition_of_values:
                       s += "{{$\\bs{{{}}}$}}".format(int(value))
                    else: s += "{0}"
                elif value == np.inf: s += "\cellcolor{red!20}{$\\infty$}"
                else:
                    exponent = int(math.floor(math.log10(abs(value))))
                    decimal = value / (10 ** exponent)

                    # Decidimos si resaltamos el valor (HL) o no
                    content = "{0:d}".format(int(value)) if value.is_integer() else "{0:.2f}".format(value)
                    if value == best_value and value != max_intValue and not repetition_of_values: 
                        # Lógica para el MEJOR valor
                        if -1 <= exponent <= 3: s += "{{$\\bs{{{}}}$}}".format(content)
                        else: s += "\\tableHL{{{0:.2f}}}{{{1}}}".format(decimal, exponent)
                    else:
                        #If the value is not to big, we can just print the value
                        if -1 <= exponent <= 3: s += "{{{}}}".format(content)
                        else: s += "{:.2f}e{}".format(decimal, exponent)

                #Separator of columns and rows
                if j != 5: s += " & "
                else: s += "\\\\"
                
            print(s)
            printed_values += 1 
    #Show the information of the printed table
    if show_info:
        print("Table variable being written:", variable_names[columnToPrint])
        print(f"In total were printed {printed_values} rows.")


#First column of results (Number of iterations)
#printRAWInfo(1, addProblemName = True, addProblemDim=True, show_info=False, dimBoundUpper = 1999) #Lowdimensional problems
#printRAWInfo(1, addProblemName = True, addProblemDim=True, show_info=False, dimBoundLower = 2000) #Highdimensional problems

#Second column of results (Last gradient)
#printRAWInfo(2, addProblemName = False, addProblemDim=False, show_info=False, dimBoundUpper=1999)
#printRAWInfo(2, addProblemName = False, addProblemDim=False, show_info=False, dimBoundLower=2000)

#Third column of results (Execution time)
#printRAWInfo(3, addProblemName = True, addProblemDim=False, show_info=False, dimBoundUpper = 1999)
#printRAWInfo(3, addProblemName = True, addProblemDim=False, show_info=False, dimBoundLower = 2000)

#Last column of results (Iterations per second)
#printRAWInfo(4, addProblemName = False, addProblemDim=False, show_info=False, dimBoundUpper = 1999)
#printRAWInfo(4, addProblemName = False, addProblemDim=False, show_info=False, dimBoundLower= 2000)

In [7]:
#print("Number of problems:", len(names_frames))

def generate_stateTable(result_matrix, names_frames, dimensions:dict, 
                        dimBoundLower: int = 0, dimBoundUpper:int = np.inf, show_dimension:bool = True, 
                        colorBoundLower:str = "ForestGreen", colorBoundUpper:str = "FireRed", alpha=0.1,
                        separation:int = 0, only_show_Randoms:bool = True, show_probs: bool = False, use_symbols:bool=True,
                        colorSymbols:bool = True, colorSymbolCheck:str = "ForestGreen", colorSymbolCross:str = "Red",
                        printfinalTable:bool = False):
    setOfRows = []
    rows_printed = []
    elementsPrinted = 0
    # Generate the body of the convergence table.
    for (i, problems_name) in enumerate(names_frames):
        
        #Get the dimension of the problem
        dimproblem = dimensions[problems_name]

        #If the problem satisfy the minimum dimension then it's printed
        if dimBoundLower <= dimproblem <= dimBoundUpper:
            s = problems_name + f" & "
            if show_dimension: s+= f"{dimproblem} &"
            for j in range(0, 6):

                #Get the value of the matrix
                value = result_matrix[0, i, j]
                
                #Add the color depending the percentage of convergence
                percentage = int(value * 100)
                s += "\cellcolor{"+colorBoundLower+f"!{percentage}!"+colorBoundUpper+f"!{int(alpha*100)}"+"}"

                #Add the probability whether it's number or symbol
                if show_probs:
                    if only_show_Randoms:
                        if value!= 0.0 and value!=1.0:
                            s += f" {{\\scriptsize {value:.1f}}}"
                    if use_symbols:
                        if not colorSymbols:
                            if value == 0.0:s += "\\texttimes"
                            elif value == 1.0:s += "\\checkmark"
                        else:
                            if value == 0.0:s += f"\\textcolor{{{colorSymbolCross}}}"+"{\\texttimes}"
                            elif value == 1.0:s += f"\\textcolor{{{colorSymbolCheck}}}"+"{\\checkmark}"
                    else:
                        s += f" {{\\scriptsize {value:.1f}}}"
                
                #Separators of colums
                if j != 5: s += " & "
                else: s += " \\\\"

            #Print the constructed row of the table
            elementsPrinted+=1
            rows_printed.append(s + "\n")
            #print(s)
            
            #To break the table into other subtable
            if separation!=0:
                if ((elementsPrinted % separation)==0):
                    if printfinalTable: 
                        print("".join(rows_printed))
                        print("------ Table Break ------")                        
                    setOfRows.append(rows_printed)
                    rows_printed = []
        else:
            pass
    if printfinalTable: print("".join(rows_printed))
    setOfRows.append(rows_printed)
    print(f"{elementsPrinted} lines has been printed.")
    return setOfRows


def printTable(rows:list[str], columns_name:list[str], style="c"):
    """Print the table using the given separation"""
    
    for setOfRows in rows:
        #HEARDER OF THE TABLE
        table_command = "\\begin{tabular}"+"{"+ style+ "}\n"
        table_command += "\\toprule\n"

        #NAME OF THE COLUMNS
        columns_rows = [columns + " & " if i != len(columns_name)-1 else columns + "\\\\ \n" for i, columns in enumerate(columns_name)]
        table_command += "".join(columns_rows)
        table_command += "\\midrule\n"
        
        #Body of the table
        table_command += "".join(setOfRows)

        #Final of the table
        table_command += "\\bottomrule\n"
        table_command += "\\end{tabular}"

        print(table_command)    
        yield


def printHoleTable(tbodys:list[str], columns_name:list[str], subtables_nummer:int,
                style:str="c", position:str = "H", addCentering:bool = True, scaleboxfactor:float = 1.0):
    
    #Assertion of scaleboxfactor
    assert 0.0 < scaleboxfactor <= 1, "Not acceptable value for scalebox, must be on interval (0,1]"
    
    #Enviroment of the tables
    print("\\begin{table}"+f"[{position}]")
    if addCentering: print("\\centering")
    print(f"\\scalebox{{{scaleboxfactor:.2f}}}"+"{")
    print("\\begin{tabular}"+"{"+ "c"*subtables_nummer + "}")

    tables = printTable(tbodys, columns_name, style)
    numberOfTables = len(tbodys)
    for i in range(1,numberOfTables+1):
        next(tables)
        if i % subtables_nummer == 0: print("\\\\")
        else: print(" & ")
    print("\\end{tabular}}")
    print("\\end{table}")

#Generate table (All problems without probs)
# tables = generate_stateTable(result_matrix, names_frames, problem_dimensions, separation= 9, show_dimension=False, 
#                     colorBoundUpper="gray", colorBoundLower="ForestGreen", alpha=0.25, 
#                     show_probs=True, use_symbols=True, colorSymbolCross="black", printfinalTable=True)

# #Generate table (Low dimensional problems 0-1999)
# tables = generate_stateTable(result_matrix, names_frames, problem_dimensions, show_dimension=False, dimBoundUpper=1999, separation=10, 
#                     colorBoundUpper="gray", colorBoundLower="ForestGreen", alpha=0.25, 
#                     show_probs=True, use_symbols=True, colorSymbolCross="black", printfinalTable=True)

#Generate table (High dimensional problems 2000-\inf)
table =  generate_stateTable(result_matrix, names_frames, problem_dimensions, show_dimension = False, dimBoundLower=2000, separation=0, 
                    colorBoundUpper="gray", colorBoundLower="ForestGreen", alpha=0.25, 
                    show_probs=True, use_symbols=True, colorSymbolCross="black", printfinalTable=True)


DIXON3DQ & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} \\
EIGENALS & \cellcolor{ForestGreen!0!gray!25}\textcolor{black}{\texttimes} & \cellcolor{ForestGreen!0!gray!25}\textcolor{black}{\texttimes} & \cellcolor{ForestGreen!0!gray!25}\textcolor{black}{\texttimes} & \cellcolor{ForestGreen!0!gray!25}\textcolor{black}{\texttimes} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} \\
EIGENBLS & \cellcolor{ForestGreen!0!gray!25}\textcolor{black}{\texttimes} & \cellcolor{ForestGreen!0!gray!25}\textcolor{black}{\text

In [9]:
#Print the table for the added LS results
printHoleTable(table, ["Fname"]+column_names_addedLS, 3, style="l"+"|c"*6+"|", scaleboxfactor=0.6)

#Print the compelete table 
#printHoleTable(tables, ["Fname"]+col_names_LS, 3, style="l"+"|c"*6+"|", scaleboxfactor=0.6)

\begin{table}[H]
\centering
\scalebox{0.60}{
\begin{tabular}{ccc}
\begin{tabular}{l|c|c|c|c|c|c|}
\toprule
Fname & G & A & Q & R & B & N\\ 
\midrule
DIXON3DQ & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} \\
EIGENALS & \cellcolor{ForestGreen!0!gray!25}\textcolor{black}{\texttimes} & \cellcolor{ForestGreen!0!gray!25}\textcolor{black}{\texttimes} & \cellcolor{ForestGreen!0!gray!25}\textcolor{black}{\texttimes} & \cellcolor{ForestGreen!0!gray!25}\textcolor{black}{\texttimes} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen}{\checkmark} & \cellcolor{ForestGreen!100!gray!25}\textcolor{ForestGreen

# Creation of a summary table


In [10]:
def generate_SummaryFloatVal(name_frmaes, cellcolors:list[str], max_default_values:list[any], verbose:int = 0):

    #Initial values
    indices_values = [[], [], [], [], []]
    printed_values = 0

    #Move throught the problem name
    for (i, problem) in enumerate(name_frmaes):
        s = problem + " & "
        #Move throught all the 5 variables (convergence don't is being used)
        for w in range(1, 5):
            if w in [1, 2, 3]:
                best_value_index = result_matrix[w, i, :].argmin()
            else:
                best_value_index = result_matrix[w, i, :].argmax()
            if w != 4:
                s += cellcolors[best_value_index] + " & "
            else:
                s += cellcolors[best_value_index]
            if result_matrix[w, i, best_value_index] != max_default_values[w]:
                indices_values[w].append(int(best_value_index))
        s += "\\\\"
        print(s)
        printed_values += 1
    
    #Information
    if verbose == 0: print("Total lines printed", printed_values)


#Printed values (it should follow the given order in the result matrix)
cellcolors = ["\\cellcolor{blue!20}A", "\\cellcolor{red!20}G", "\\cellcolor{purple!20}R", "\\cellcolor{green!20}N", "\\cellcolor{yellow!20}B"]
cellcolors_LS =  ["\\cellcolor{orange!20}S", "\\cellcolor{blue!20}A", "\\cellcolor{red!20}Q", "\\cellcolor{purple!20}R", "\\cellcolor{yellow!20}B", "\\cellcolor{green!20}N"]
default_values = [None, 1000, 0, np.inf, np.inf, np.inf, ]

#Print the table
generate_SummaryFloatVal(names_frames, cellcolors_LS, default_values, 0)
    

ARGLINA & \cellcolor{orange!20}S & \cellcolor{orange!20}S & \cellcolor{orange!20}S & \cellcolor{orange!20}S\\
BARD & \cellcolor{red!20}Q & \cellcolor{blue!20}A & \cellcolor{purple!20}R & \cellcolor{orange!20}S\\
BEALE & \cellcolor{yellow!20}B & \cellcolor{blue!20}A & \cellcolor{green!20}N & \cellcolor{orange!20}S\\
BRKMCC & \cellcolor{purple!20}R & \cellcolor{blue!20}A & \cellcolor{purple!20}R & \cellcolor{purple!20}R\\
BROWNAL & \cellcolor{blue!20}A & \cellcolor{green!20}N & \cellcolor{green!20}N & \cellcolor{orange!20}S\\
BROWNBS & \cellcolor{green!20}N & \cellcolor{blue!20}A & \cellcolor{green!20}N & \cellcolor{purple!20}R\\
BROWNDEN & \cellcolor{green!20}N & \cellcolor{green!20}N & \cellcolor{green!20}N & \cellcolor{green!20}N\\
CHNROSNB & \cellcolor{green!20}N & \cellcolor{green!20}N & \cellcolor{yellow!20}B & \cellcolor{orange!20}S\\
CLIFF & \cellcolor{blue!20}A & \cellcolor{red!20}Q & \cellcolor{purple!20}R & \cellcolor{orange!20}S\\
CUBE & \cellcolor{blue!20}A & \cellcolor{purp

In [11]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Map values to your specific names
# Replace 'Name A', etc., with your actual category names
sns.set_theme(style="whitegrid")
name_map = {
    0: 'Oviedo',
    1: 'Gradients',
    2: 'Random',
    3: 'Newton',
    4: 'BFGS'
}
varibles = ["Iterations", "Last Gradient", "Execution time", "Iterations per second"]


# 3. Convert to Long-Format DataFrame and apply the names
data_list = []
for i, sublist in enumerate(indices_values[1:]):
    for value in sublist:
        data_list.append({
            'Element': varibles[i], 
            'Label': name_map[value]  # Use the name instead of the number
        })
df = pd.DataFrame(data_list)

# 4. Update the color dictionary to use the new names
color_dict = {
    'Oviedo': 'blue',
    'Gradients': 'red',
    'Random': 'purple',
    'Newton': 'green',
    'BFGS': 'orange'
}

# 5. Plotting
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

ax = sns.countplot(
    data=df, 
    x='Element', 
    hue='Label', 
    palette=color_dict,
    hue_order=list(name_map.values()) # Keeps categories in order A -> E
)

plt.title('Frequency per Position with Custom Category Names', fontdict={"fontsize":12}, fontweight='bold', loc="left")
plt.xlabel('Measuraments', fontdict={"fontsize":8})
plt.ylabel('')
plt.grid(alpha=0.8)
plt.legend(title='Methods', loc='upper left')
plt.tight_layout()

#Save plot (For Poster)
#plt.savefig("images/countplot_solvedProblems.jpg", dpi=1200)

#Save plot (For thesis)
#plt.savefig("images/countplot_solvedProblems.svg")


NameError: name 'indices_values' is not defined